<a href="https://colab.research.google.com/github/YakovDeLeon/ML_Course_PT/blob/main/Hometasks/PT_HT3_WhoIsTalking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание 3

В этом задании мы предлагаем вам не только улучшить [бейзлайн, который мы дали вам на занятии](https://colab.research.google.com/drive/1DirkE2idWXfl7TqTF0ZxBi6M37LZ0NGD?usp=sharing), но и продумать задание с точки зрения ML System Design. Ответы на вопросы, которые мы поставим перед вами, могут повлиять на код (метрики, модели и так далее).

Вам предстоит спроектировать систему классификации TCP-потоков по приложению/сервису на основе усечённой последовательности длин пакетов (до 30 значений, с направлением).

Представьте, что вы не просто обучаете модель, а проектируете промышленное решение, которое будет использоваться в корпоративной среде.

### Важно!

Сначала почитайте и изучите ноутбук и код с занятия.

# 1. Бизнес-контекст и целеполагание

Вопрос:
Зачем и кому может быть нужна система классификации приложений по зашифрованному трафику? Как и где она будет использоваться?

В ответе опишите:

- Кто является заказчиком (SOC-команда, отдел ИБ, провайдер, корпорация, государственная структура и т.д.)?

-  Какие реальные бизнес-задачи она решает?

- В каком режиме работает система:

офлайн-анализ (батч),

near-real-time,

real-time (inline)?

- Какие требования к:

задержке (latency),

пропускной способности,

интерпретируемости,

объяснимости,

обновляемости модели?

- Какие риски есть при ошибках классификации (FP/FN)?


# Ответ

Классификация приложений по зашифрованному трафику нужна в первую очередь для систем съёма трафика, в которых необходимо определить какие сервисы посещает пользователь и ограничивать доступ к нежелательному контенту со стороны системного администратора и блокировать различные кибер угрозы. Системы съёма трафика могут быть как активными, так и пассивными. Активные системы, например файрволл, встраиваются по середине между отправителем и получателем и могут модфифицировать оригинальные пакеты и блокировать трафик в момент его прихода, поэтому относятся к системам жёсткого реального времени. При пассивном съёме, например DPI, в систему попадает копия уже запроксированного трафика. Такие системы используются в основном для анализа трафика и выявления в нём различных аномалий. Так как в такую систему попадает уже копия трафика требования к задержкам тут ниже, но всё же хотелось выявить атаку как можно быстрее. Поэтому такие системы относятся к системам мягкого реального времени.

Для определённости, в рамках текущей задачи попытаемся встроить классификацию приложений в зашифрованном траффике в файрвол. Файрвол обычно поддерживает MITM и умеет дешифровать трафик, однако весь трафик обычно не дешифруются, так как это требует огромных вычислительных ресурсов, поэтому через файрвол TLS идёт в соотношении: 30% - расшифровывается, 70% - нет. Зашифрованный трафик файрвол уже умеет классифицировать по SNI, который передаётся при установлении соединения. Однако злоумышленник может использовать фейковый SNI и тем самым обходить политики безопасности.

Исходя из этого требования к модели следующие:
* Анализ в режиме real-time, любые задержки могут быть критичные для пользователя
* Latency: Модель должна принимать решение не позднее чем через 4кб для каждой из сторон, или суммарно 8 кб.
* В иделае модель должна быть легковесной и не должна влиять на пропускную способность, как будет на практике сказать сложно, но не хотелось допустить сильную деградацию производительности
* Интерпретируемость: Решение модели должно быть трассируемым до набора входных признаков.
* Модель должна содержать информацию о принятом решении, чтобы SOC команда смогла отличить правильное решение модели от ложного срабатывания
* Модель должна обновляться ежедневно, чтобы оперативно реагировать на новые угрозы. Обновление модели не должно приводить к перезапуску NGFW и влиять на обработку DP данных
* При ложном срабатывании может блокироваться легитимный трафик, поэтому должна быть надстройка на СУ, позволяющая отключить учёт результата модели



# 2. Данные и обработка: классы, дисбаланс, новые сервисы

Вопрос:
- Как нужно подготовить данные и организовать работу с классами, учитывая сильный дисбаланс?

В ответе раскройте:

- Нужно ли:

агрегировать редкие классы в "Other"?

удалять самые редкие классы?


- Как дисбаланс повлияет на:

обучение,

переобучение,

выбор метрики,

интерпретацию результатов?


- Какие стратегии борьбы с дисбалансом вы бы применили?

- Как система должна реагировать на новые приложения, которых не было в обучающей выборке?

- Нужно ли учитывать изменение поведения сервисов со временем (concept drift)?

# Ответ:

Для ngfw 20% от известных приложений будет составлять примерно 80% всего трафика. На остальные 70% приложений придётся, проценты или доли процентов трафика и ещё 10% попадут под нулевые срабатывания.

Поэтому очень редкие классы будет уместно аггрегировать в какую-то более общую категорию. Те классы, которые не представляют угрозы и имеют мало примеров трафика имеет смысл объединить в "Unknown application". Классы, которые являются редкими, но критичными к безопасности (например VPN, TOR, C2) нужно сохранять.

Самые редкие классы имееть смысл удалять, если:
* класс является шумовым
* плохо размечен
* имеет слишком мало данных
* не является значимым для security

В идеале такие классы лучше агрегировать в один или переводить в "Unknown applications".

Посколько самые популярные классы всегда будут составлять большую долю трафика возникает дисбаланс. Поэтому при обучении важно вносить корректировки в модель, иначе модель:
* будет минимизировать общую ошибку
* будет игнорировать миноритарные классы

Пример:

YouTube — 60%
Slack — 20%
Zoom — 10%
VPN — 0.5%
TOR — 0.1%

Модель может полностью игнорировать TOR и всё равно иметь 99% accuracy.

При таком дисбалансе, существует большой риск, что миноритарные классы:
* легко переобучаются
* модель просто запоминает конкретные TLS-потоки
* плохая генерализация (плохая обобщающая способность)

При выборре метрики общая accuracy будет работать плохо, так как мы не будем видеть какие классы путаем между собой. Например ютуб будет угадываться всегда, VPN - нет, но accuracy будет высокий. Поэтому в этом случае лушче исопльзовать:
* матрицу предсказаний (precision, recall)
* f1-меру
* каждую из этих метрик смотрим по сработке на каждый конкретный класс

Для борбы с дисбалансом будем использовать взвешивание классов. Для более редких будем назначать более высокий способ. Самый простой способ в качестве веса взять 1 / частоту встречания (других пока не знаю:-)). Однако надо понимать, что на разных предприятих, где установлен NGFW, может быть разный трафик, поэтому здесь хорошо бы собрать данных с разных мест.

Чатгпт ещё предлагает попробовать иерархическую классификацию, но это может увеличить сложность вычислений и привести к большей деградации, поскольку несколько моделей надо запускать по несколько раз на одном и том же наборе данных. Это может быть особенно заметно, если мы захотим также гонять содержимое зашифрованного трафика (хотя пока сложно сказать, насколько в этом есть смысл). Точно сказать, как себя поведёт модель тут сложно, надо исследовать.

При появлении новых приложений, модель может относить их к уже известным классам, что может быть не очень хорошо, так как новые VPN будут классифицироваться например как YouTube. Поэтому для контроля для с этим чатгпт советует:
* Threshold по confidence. Если max probability < T → Unknown
* Energy-based detection
* Distance to class centroid
* One-class classifier поверх эмбеддингов

Моделью должен обязательно учитываться concept-drift. Так как большинство приложений используют в основном REST API поверх HTTP внутри TLS, то взаимодействие между клеинтом и сервером может меняться со времени из-за обновления протокола взаимодействия. Это может отразиться на разных размерах пакетов, на разных урлах внутри процедуры обмена сообщения, если в каждом регионе размещён свой сервер.

Исходя из этого drift может быть:
* Feature drift (изменились количество данных отправялемых клиентом и сервером за тот же промежуток наблюдения)
* Prior drift (изменилось распределение классов, например из-за замедления ютуба, в топ может вылезти рутуб)
* Label drift (класс изменил поведение, (сомнительно, но окэй, не придумал пример такого))

Что с этим делать:
* Continuous monitoring
  * Track per-class recall over time
  * Track feature distribution
* Периодическое переобучение
  * Раз в день/месяц
  * Или при детекте деградации
  * Но тут вопрос какие данны подавать для переобчения, вряд ли снимать весь трафик на терабайты и подавать его на переобучение будет хорошей идеей. Наверное тут правильнее будет отдавать трафик с данными, которые попали под категорию "Unknown Applictaion"

* Shadow validation
  * Новая модель проверяется параллельно старой.


# 3. Метрика и выбор алгоритмов

Вопрос:
Какую метрику качества вы выберете и почему? Нужно ли ограничивать себя определёнными алгоритмами?


- Выберите и обоснуйте метрику.

- Объясните, почему выбранная метрика соответствует бизнес-целям.

- Нужно ли использовать несколько метрик?

- Допустимы ли любые модели? Есть ли ограничения?

# Ответ

Как уже было сказано ранее, у нас имеется сильный дисбаланс классов, при этом NGFW должен не пропускать угрозы (то есть должен быть высокий recall) и не блокировать легитимный трафик (должен быть высокий precision). F1 является компроммисом между ними. Но для информативности будем использовать все три метрики:
* Для оценки безопасности, используем recall, так как среди всего трафика мы должны блокировать как можно больше того, что попадает под угрозы
* Для оценки стабильности используем Precision, чтобы понимать насколько можно доверять результату
* Для оценки общей устойчивости, будем использовать f-score

Модель не должна оказывать сильного влияния на обработку сетевого трафика. Поэтому ма должны ориентироваться на максимально лёгкие модели, например:
* Logistic Regression
* Gradient Boosted Trees (ограниченной глубины)


# 4. Персональные данные, приватность и деплой

Вопрос:
Считается ли такой датасет персональными данными? Какие риски существуют?


- Можно ли по последовательности размеров пакетов деанонимизировать пользователя?

- Где может быть развернута система:

on-premise,

в облаке,

на edge-устройствах?

- Какие меры нужно предусмотреть:

анонимизация,

логирование,

ограничение хранения,

контроль доступа?

# Ответ!

Да такой датасет является персональными данными. В худшем случае если соотнести размеры пакетов с трафиком, где они были они добыты, то можно узнать ip пользователя. По нему найти весь его трафик, а там если много вычислительных ресурсов можно и весь трафик попытаться дешифровать.

Система должна быть развёрнута on-permise. Для обучения модели трафик можно сгенерировать своими силами внутри компании, поэтому нет смысла выносить систему куда-то ещё.

Так как модель не покидает периметра компании, то кажется кроме контроля доступа здесь дополнительно ничего не треубется.

# Улучшение бейзлайна.

Наконец, вы можете писать код!

С учетом выбранной метрики и выбранного пула алгоритмов, улучшите бейзлайн с занятия:

- Поработайте с признаками - придумайте новые, затем попробуйте отбор признаков
- Поработайте с разными моделями, подбирайте у них гиперпараметры
- Можете попробовать использовать кластеризацию для извлечения дополнительных признаков

<a href="https://colab.research.google.com/github/YakovDeLeon/ML_Course_PT/blob/main/Lecture4_Clustering%26MLSD/PT_Practice4_WhoIsTalking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Сначала скачаем данные

In [1]:
!wget -O response.json \
"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key=https://disk.yandex.ru/d/7LCvNsGL1mv90Q"

--2026-03-04 08:37:00--  https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key=https://disk.yandex.ru/d/7LCvNsGL1mv90Q
Resolving cloud-api.yandex.net (cloud-api.yandex.net)... 213.180.204.127, 2a02:6b8::1:127
Connecting to cloud-api.yandex.net (cloud-api.yandex.net)|213.180.204.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 617 [application/json]
Saving to: ‘response.json’

response.json       100%[===================>]     617  --.-KB/s    in 0s      

2026-03-04 08:37:02 (36.8 MB/s) - ‘response.json’ saved [617/617]



In [2]:
import json

with open("response.json") as f:
    data = json.load(f)

print(data["href"])

https://downloader.disk.yandex.ru/disk/2cc0bb58fa52381e1088248dd2070aede68759a73fdeee494c05b7efd2022d65/69a8276e/fKqInKw3d7bLFOeFnMGnhOEt74e_AcHrfoeOiWQFqnxSx2KlJFe3moqi-rTFCO6djYu3q6EHAUP4X1BQu7yk7eyLDbl9lafVd4ruaDAPlxGr8npumZHI4midPdWhecNq?uid=0&filename=whos-talking-classify-the-app-by-its-packets.zip&disposition=attachment&hash=m8VLCd1u1YCGUik9dZVkJjbxXRNbrE0kS1p%2BEIBguw1klnm56L9GDaExZ6Yk0MeDq/J6bpmRyOJonT3VoXnDag%3D%3D%3A&limit=0&content_type=application%2Fzip&owner_uid=259847718&fsize=305374529&hid=11ebcd4213988e57481af625f75bf49e&media_type=compressed&tknv=v3


In [4]:
!wget -O archive.zip "https://downloader.disk.yandex.ru/disk/2cc0bb58fa52381e1088248dd2070aede68759a73fdeee494c05b7efd2022d65/69a8276e/fKqInKw3d7bLFOeFnMGnhOEt74e_AcHrfoeOiWQFqnxSx2KlJFe3moqi-rTFCO6djYu3q6EHAUP4X1BQu7yk7eyLDbl9lafVd4ruaDAPlxGr8npumZHI4midPdWhecNq?uid=0&filename=whos-talking-classify-the-app-by-its-packets.zip&disposition=attachment&hash=m8VLCd1u1YCGUik9dZVkJjbxXRNbrE0kS1p%2BEIBguw1klnm56L9GDaExZ6Yk0MeDq/J6bpmRyOJonT3VoXnDag%3D%3D%3A&limit=0&content_type=application%2Fzip&owner_uid=259847718&fsize=305374529&hid=11ebcd4213988e57481af625f75bf49e&media_type=compressed&tknv=v3"
# !wget -O archive.zip str(data["href"])

--2026-03-04 08:37:40--  https://downloader.disk.yandex.ru/disk/2cc0bb58fa52381e1088248dd2070aede68759a73fdeee494c05b7efd2022d65/69a8276e/fKqInKw3d7bLFOeFnMGnhOEt74e_AcHrfoeOiWQFqnxSx2KlJFe3moqi-rTFCO6djYu3q6EHAUP4X1BQu7yk7eyLDbl9lafVd4ruaDAPlxGr8npumZHI4midPdWhecNq?uid=0&filename=whos-talking-classify-the-app-by-its-packets.zip&disposition=attachment&hash=m8VLCd1u1YCGUik9dZVkJjbxXRNbrE0kS1p%2BEIBguw1klnm56L9GDaExZ6Yk0MeDq/J6bpmRyOJonT3VoXnDag%3D%3D%3A&limit=0&content_type=application%2Fzip&owner_uid=259847718&fsize=305374529&hid=11ebcd4213988e57481af625f75bf49e&media_type=compressed&tknv=v3
Resolving downloader.disk.yandex.ru (downloader.disk.yandex.ru)... 77.88.21.127, 2a02:6b8::2:127
Connecting to downloader.disk.yandex.ru (downloader.disk.yandex.ru)|77.88.21.127|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://s02klg.storage.yandex.net/rdisk/2cc0bb58fa52381e1088248dd2070aede68759a73fdeee494c05b7efd2022d65/69a8276e/fKqInKw3d7bLFOeFnMGnhOEt74e_Ac

In [5]:
!unzip archive.zip -d data

Archive:  archive.zip
  inflating: data/sample_submission.csv  
  inflating: data/test.csv           
  inflating: data/train.csv          


Установим необходимые библиотеки

In [6]:
!pip install catboost -q

import numpy as np
import pandas as pd
import gc

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier, Pool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


Загрузим данные и посмотрим на них

In [7]:
df = pd.read_csv('/content/data/train.csv')

df.info()

/tmp/ipykernel_224/525312853.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/data/train.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8248546 entries, 0 to 8248545
Data columns (total 31 columns):
 #   Column       Dtype  
---  ------       -----  
 0   app_service  object 
 1   tcp_len_1    int64  
 2   tcp_len_2    float64
 3   tcp_len_3    float64
 4   tcp_len_4    float64
 5   tcp_len_5    float64
 6   tcp_len_6    float64
 7   tcp_len_7    float64
 8   tcp_len_8    float64
 9   tcp_len_9    float64
 10  tcp_len_10   float64
 11  tcp_len_11   float64
 12  tcp_len_12   float64
 13  tcp_len_13   float64
 14  tcp_len_14   float64
 15  tcp_len_15   float64
 16  tcp_len_16   float64
 17  tcp_len_17   float64
 18  tcp_len_18   float64
 19  tcp_len_19   float64
 20  tcp_len_20   float64
 21  tcp_len_21   float64
 22  tcp_len_22   float64
 23  tcp_len_23   float64
 24  tcp_len_24   float64
 25  tcp_len_25   float64
 26  tcp_len_26   float64
 27  tcp_len_27   float64
 28  tcp_len_28   float64
 29  tcp_len_29   float64
 30  tcp_len_30   float64
dtypes: float64(29), int64(1)

In [8]:
df.sample(5)

,app_service,tcp_len_1,tcp_len_2,tcp_len_3,tcp_len_4,tcp_len_5,tcp_len_6,tcp_len_7,tcp_len_8,tcp_len_9,...,tcp_len_21,tcp_len_22,tcp_len_23,tcp_len_24,tcp_len_25,tcp_len_26,tcp_len_27,tcp_len_28,tcp_len_29,tcp_len_30
5232697,kontur,1404,415.0,-1448.0,-1448.0,-1200.0,-333.0,80.0,92.0,398.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4424823,google,1412,880.0,-1412.0,-60.0,74.0,92.0,735.0,241.0,-976.0,...,-99.0,-31.0,-39.0,35.0,39.0,79.0,183.0,-70.0,-77.0,-31.0
7901659,yandex_mail,1410,1108.0,-1220.0,-1220.0,-1220.0,-436.0,-1118.0,80.0,-287.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5962866,mts,1460,326.0,-1460.0,-953.0,64.0,92.0,497.0,-271.0,-271.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2161936,Rutube,1410,676.0,-260.0,80.0,1410.0,1241.0,-287.0,-550.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
df.memory_usage(deep=True).sum() / 1024**2

np.float64(2317.443407058716)

Казалось бы по определению длина - целая величина, а тут используеться float64, давайте экономить память. Проверим диапозон значений, найдем есть ли наны или инфы, проверим количество не целых чисел

In [10]:
target_col = "app_service"
feat_cols = [c for c in df.columns if c.startswith("tcp_len_")]
print(feat_cols)

['tcp_len_1', 'tcp_len_2', 'tcp_len_3', 'tcp_len_4', 'tcp_len_5', 'tcp_len_6', 'tcp_len_7', 'tcp_len_8', 'tcp_len_9', 'tcp_len_10', 'tcp_len_11', 'tcp_len_12', 'tcp_len_13', 'tcp_len_14', 'tcp_len_15', 'tcp_len_16', 'tcp_len_17', 'tcp_len_18', 'tcp_len_19', 'tcp_len_20', 'tcp_len_21', 'tcp_len_22', 'tcp_len_23', 'tcp_len_24', 'tcp_len_25', 'tcp_len_26', 'tcp_len_27', 'tcp_len_28', 'tcp_len_29', 'tcp_len_30']


In [11]:
stats = {
    "min_value": df[feat_cols].min().min(),
    "max_value": df[feat_cols].max().max(),
    "nan_count": np.isnan(df[feat_cols].values).sum(),
    "inf_count": np.isinf(df[feat_cols].values).sum(),
    "non_integer_count": np.sum((df[feat_cols].values % 1) != 0),
    "zero_value": np.sum((df[feat_cols].values == 0)),
}

stats

{'min_value': -1464.0,
 'max_value': 1460.0,
 'nan_count': np.int64(0),
 'inf_count': np.int64(0),
 'non_integer_count': np.int64(0),
 'zero_value': np.int64(75589496)}

Как видно значения по в диапозоне от -1464 до 1460 нет ни нанов ни инфов ни дробных чисел - изменим тип на int16

In [12]:
feat_cols = [c for c in df.columns if c.startswith("tcp_len_")]
df[feat_cols] = df[feat_cols].astype(np.int16)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8248546 entries, 0 to 8248545
Data columns (total 31 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   app_service  object
 1   tcp_len_1    int16 
 2   tcp_len_2    int16 
 3   tcp_len_3    int16 
 4   tcp_len_4    int16 
 5   tcp_len_5    int16 
 6   tcp_len_6    int16 
 7   tcp_len_7    int16 
 8   tcp_len_8    int16 
 9   tcp_len_9    int16 
 10  tcp_len_10   int16 
 11  tcp_len_11   int16 
 12  tcp_len_12   int16 
 13  tcp_len_13   int16 
 14  tcp_len_14   int16 
 15  tcp_len_15   int16 
 16  tcp_len_16   int16 
 17  tcp_len_17   int16 
 18  tcp_len_18   int16 
 19  tcp_len_19   int16 
 20  tcp_len_20   int16 
 21  tcp_len_21   int16 
 22  tcp_len_22   int16 
 23  tcp_len_23   int16 
 24  tcp_len_24   int16 
 25  tcp_len_25   int16 
 26  tcp_len_26   int16 
 27  tcp_len_27   int16 
 28  tcp_len_28   int16 
 29  tcp_len_29   int16 
 30  tcp_len_30   int16 
dtypes: int16(30), object(1)
memory usage: 534.9+ MB


In [13]:
df.memory_usage(deep=True).sum() / 1024**2

np.float64(901.4866428375244)

Почти в 4 раза экономнее, RAM в колабе скажет спасибо

In [14]:
N = len(df)
N

8248546

In [15]:
X0 = df[feat_cols]
y = df[target_col].astype(str)

sample_n = 1_000_000 # всё не влезет
idx = np.random.RandomState(42).choice(N, size=sample_n, replace=False)

X0 = X0.iloc[idx]
y  = y.iloc[idx]

Посмотрим на целевую переменную

In [16]:
y.value_counts()

,count
app_service,
Telegram,12315
287,12277
GMail,12277
Wildberries,12268
Teams,12266
...,...
apteka-ru,135
appdynamics,133
MEGA,131


In [17]:
set(y)

{'1',
 '1-password',
 '101-radio',
 '113',
 '120',
 '15',
 '218',
 '287',
 '299',
 '2gis',
 '343',
 '4',
 '93',
 'Amazon',
 'Anydesk',
 'Apple',
 'Baidu',
 'DataSaver',
 'DoH_DoT',
 'DropBox',
 'Facebook',
 'GMail',
 'Github',
 'Gitlab',
 'Instagram',
 'Ktalk',
 'LinkedIn',
 'Livejournal',
 'MEGA',
 'MSN',
 'Mattermost',
 'Odnoklassniki',
 'Ozon',
 'PlayStore',
 'Playstation',
 'QQ',
 'Quora',
 'Reddit',
 'Rutube',
 'Skype',
 'Snapchat',
 'SoundCloud',
 'Spotify',
 'Steam',
 'Teams',
 'Telegram',
 'TikTok',
 'Twitch',
 'Twitter',
 'WhatsApp',
 'Wikipedia',
 'Wildberries',
 'Yandex',
 'YouTube',
 'adblock-plus',
 'adobe',
 'aeroflot-online',
 'afisha-ru',
 'alfabank',
 'ali-wangwang-file-transfer',
 'alipay',
 'amd-online',
 'andata',
 'any-run',
 'anygo',
 'appdynamics',
 'apple-community',
 'apple-siri',
 'apple_icloud',
 'apple_push',
 'apteka-ru',
 'avito',
 'azure',
 'banki-ru',
 'beeline-online',
 'bing',
 'bitly',
 'blog-posting',
 'braintree',
 'bugsnag',
 'bybit',
 'ca-technolo

### Feature engineering

In [18]:
X0.head()

,tcp_len_1,tcp_len_2,tcp_len_3,tcp_len_4,tcp_len_5,tcp_len_6,tcp_len_7,tcp_len_8,tcp_len_9,tcp_len_10,...,tcp_len_21,tcp_len_22,tcp_len_23,tcp_len_24,tcp_len_25,tcp_len_26,tcp_len_27,tcp_len_28,tcp_len_29,tcp_len_30
3899906,517,-1448,-1448,-1448,-23,64,992,-179,-62,-31,...,586,31,-251,1448,54,31,-305,-31,586,31
5090771,543,-399,51,234,-238,-1460,-1460,-1209,251,-1405,...,-1460,-633,245,-1405,-1460,-1460,-7,249,-1405,-1359
6160203,517,-203,314,-1460,-349,-1136,250,-1391,-1137,250,...,0,0,0,0,0,0,0,0,0,0
3764889,517,-1448,-1448,-579,64,1208,56,-1208,-1448,-946,...,0,0,0,0,0,0,0,0,0,0
7759903,1460,647,-1460,-1460,-1176,-1287,64,92,548,-287,...,0,0,0,0,0,0,0,0,0,0


In [19]:
X = X0.copy()

pos = (X0 > 0)
neg = (X0 < 0)
nonzero = (X0 != 0)
not_zeros = (X0.iloc[:, :-1].values != 0) & (X0.iloc[:, 1:].values != 0)
switches = (pos.iloc[:, :-1].values != pos.iloc[:, 1:].values)
switches_full = np.column_stack([np.zeros(switches.shape[0], dtype=bool), switches])

actual_switches = (switches_full & nonzero)
group_ids = np.cumsum(actual_switches, axis=1)
first_request = np.where(group_ids == 0, X0.values, 0)
first_response = np.where(group_ids == 1, X0.values, 0)
second_request = np.where(group_ids == 2, X0.values, 0)
second_response = np.where(group_ids == 3, X0.values, 0)
third_request = np.where(group_ids == 4, X0.values, 0)
third_response = np.where(group_ids == 5, X0.values, 0)
fourth_request = np.where(group_ids == 6, X0.values, 0)
fourth_response = np.where(group_ids == 7, X0.values, 0)
fifth_request = np.where(group_ids == 8, X0.values, 0)
fifth_response = np.where(group_ids == 9, X0.values, 0)

X["pkt_cnt"] = nonzero.sum(axis=1).astype(np.int16) # общее количество пакетов в потоке
X["n_up"]    = pos.sum(axis=1).astype(np.int16) # количество пакетов от клиента
X["n_down"]  = neg.sum(axis=1).astype(np.int16) # количество пакетов от сервера

X["bytes_up"]   = X0.where(pos, 0).sum(axis=1).astype(np.int32) # суммарное количество байт, отправленных клиентом
X["bytes_down"] = (-X0.where(neg, 0)).sum(axis=1).astype(np.int32) # суммарное количество байт, полученных от сервера
X["total_switches"] = (actual_switches.sum(axis=1)).astype(np.int16) # количество смен направлений
X["first_request"] = np.sum(first_request, axis=1)
X["first_response"] = np.sum(first_response, axis=1)
X["second_request"] = np.sum(second_request, axis=1)
X["second_response"] = np.sum(second_response, axis=1)
X["third_request"] = np.sum(third_request, axis=1)
X["third_response"] = np.sum(third_response, axis=1)
X["fourth_request"] = np.sum(fourth_request, axis=1)
X["fourth_response"] = np.sum(fourth_response, axis=1)
X["fifth_request"] = np.sum(fifth_request, axis=1)
X["fifth_response"] = np.sum(fifth_response, axis=1)


print(X["total_switches"].min().min())
print(X["total_switches"].max().max())
print(X["total_switches"].mode()[0])


print(X.iloc[0])
print(switches_full[0])

0
29
5
tcp_len_1           517
tcp_len_2         -1448
tcp_len_3         -1448
tcp_len_4         -1448
tcp_len_5           -23
tcp_len_6            64
tcp_len_7           992
tcp_len_8          -179
tcp_len_9           -62
tcp_len_10          -31
tcp_len_11          -35
tcp_len_12           31
tcp_len_13         -251
tcp_len_14           31
tcp_len_15         1448
tcp_len_16          162
tcp_len_17          -35
tcp_len_18           31
tcp_len_19         -305
tcp_len_20          -31
tcp_len_21          586
tcp_len_22           31
tcp_len_23         -251
tcp_len_24         1448
tcp_len_25           54
tcp_len_26           31
tcp_len_27         -305
tcp_len_28          -31
tcp_len_29          586
tcp_len_30           31
pkt_cnt              30
n_up                 15
n_down               15
bytes_up           6043
bytes_down         5883
total_switches       14
first_request       517
first_response    -4367
second_request     1056
second_response    -307
third_request        31
third_res

### Обучение модели и валидация

In [20]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    auto_class_weights='Balanced',
    # custom_metric=["Precision", "Recall", "Accuracy"],
    iterations=250,
    depth=10,
    learning_rate=0.15,
    random_seed=42,
    verbose=True,
    task_type="GPU",
    devices="0"
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

pred = model.predict(X_val).reshape(-1)
print("val accuracy:", accuracy_score(y_val, pred))

0:	learn: 0.2249258	test: 0.2169201	best: 0.2169201 (0)	total: 10.3s	remaining: 42m 42s
1:	learn: 0.2480010	test: 0.2431540	best: 0.2431540 (1)	total: 18.7s	remaining: 38m 41s
2:	learn: 0.2714840	test: 0.2651989	best: 0.2651989 (2)	total: 27.5s	remaining: 37m 42s
3:	learn: 0.3422502	test: 0.3357996	best: 0.3357996 (3)	total: 37.9s	remaining: 38m 53s
4:	learn: 0.3624045	test: 0.3589908	best: 0.3589908 (4)	total: 46.2s	remaining: 37m 41s
5:	learn: 0.3813159	test: 0.3766863	best: 0.3766863 (5)	total: 55.2s	remaining: 37m 24s
6:	learn: 0.4117643	test: 0.4043336	best: 0.4043336 (6)	total: 1m 5s	remaining: 37m 56s
7:	learn: 0.4441605	test: 0.4333640	best: 0.4333640 (7)	total: 1m 13s	remaining: 37m 4s
8:	learn: 0.4721334	test: 0.4609169	best: 0.4609169 (8)	total: 1m 22s	remaining: 36m 47s
9:	learn: 0.4967259	test: 0.4846353	best: 0.4846353 (9)	total: 1m 32s	remaining: 36m 51s
10:	learn: 0.5268994	test: 0.5093865	best: 0.5093865 (10)	total: 1m 41s	remaining: 36m 51s
11:	learn: 0.5423320	test: 

In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_val, pred))

                                   precision    recall  f1-score   support

                                1       0.88      0.80      0.84      2414
                       1-password       0.47      0.76      0.58        62
                        101-radio       0.69      0.74      0.71        27
                              113       0.97      0.93      0.95      2416
                              120       0.88      0.87      0.88      2417
                               15       0.76      0.97      0.85       538
                              218       0.91      0.95      0.93        82
                              287       0.93      0.97      0.95      2455
                              299       0.92      0.92      0.92      2453
                             2gis       0.96      0.92      0.94      1821
                              343       0.51      0.57      0.54      2437
                                4       0.88      0.89      0.89      2426
                        